In [64]:
import itertools

q = 2
n = 3

G = GL(n, GF(q))
R.<v> = LaurentPolynomialRing(QQ)

def is_upper_triangular(a):
    m = matrix(a)
    return all(m[i, j] == 0 for i in range(n) for j in range(i))

def unip_upper_triang(a):
    m = matrix(a)  # Convert the group element to a Sage matrix
    # Check for upper triangularity and diagonal entries
    upper_triangular = all(m[i, j] == 0 for i in range(n) for j in range(i))
    diagonal_ones = all(m[i, i] == 1 for i in range(n))
    return upper_triangular and diagonal_ones

def is_permutation(a):
    m = matrix(a)
    isperm = True
    for i in range(n):
        numones = 0
        numzeros = 0
        for j in range(n):
            if m[i, j] == 1:
                numones += 1
            elif m[i, j] == 0:
                numzeros += 1
        if numones != 1 or numzeros != n - 1:
            isperm = False
    return isperm

B = [a for a in G if is_upper_triangular(matrix(a))]
U = [a for a in G if unip_upper_triang(matrix(a))]
W = [a for a in G if is_permutation(matrix(a))]

def all_matrices(n, q):
    entries = range(q)
    matrices = list(itertools.product(entries, repeat=n*n))
    matrices = [ matrix([list(m[i*n : (i+1)*n]) for i in range(n)]) for m in matrices ]
    return matrices

def gl(n, q):
    # Generate all matrices
    matrices = all_matrices(n, q)
    # Filter out matrices with zero determinant modulo q
    non_singular_matrices = [m for m in matrices if m.det() % q != 0]
    return non_singular_matrices

orbit_dict = {}
for g in G:
    seen = False
    for b in B:
        if seen == False:
            a = g*b
            if a in orbit_dict:
                seen = True
                orbit_dict[a].append(g)
    if not seen:
        orbit_dict[g] = [g]

biorbit_dict = {}
for w in W:
    biorbit_dict[w] = []
for w in W:
    for u in U:
        a = u*w
        for orb in orbit_dict:
            if a in orbit_dict[orb] and orb not in biorbit_dict[w]:
                biorbit_dict[w].append(orb)

def Uw(w):
    uw = []
    for u in U:
        if w.inverse() * u * w in U:
            uw.append(u)
    return(uw)

def fixed_flags(a):
    fix = []
    for g in orbit_dict:
        #print(g.inverse() * a * g)
        #print()
        if g.inverse() * a * g in B:
            fix.append(g)
    return fix

def cap_with_w(s, w):
    #s is some subset of the keys in orbit_dict, i.e. flags
    #return the ones in the orbit of w...
    cap = []
    for x in biorbit_dict[w]:
        if x in s:
            cap.append(x)
    return cap

ordW = [W[0], W[3], W[5], W[2], W[1], W[4]]

def signature(a):
    fix = fixed_flags(a)
    sig = []
    for i in range(len(ordW)):
        if len(cap_with_w(fix, ordW[i])) == 0:
            sig.append(0)
        else:
            sig.append(v^(log(len(cap_with_w(fix, ordW[i])),2)))
    return sig

def normalize(l):
    s = 0
    newl = []
    for a in l:
        s += a
    for i in range(len(l)):
        newl.append(l[i] / s)
    return newl

def avglists(list_of_lists):
    if not list_of_lists:
        return[]
    n = len(list_of_lists)
    k = len(list_of_lists[0])
    newlist = []
    for i in range(k):
        newguy = 0
        for j in range(n):
            newguy += list_of_lists[j][i] / n
        newlist.append(newguy)
    return newlist

weights = {}
weights[Uw(ordW[0])[0]] = 0
weights[Uw(ordW[0])[1]] = 3
weights[Uw(ordW[0])[2]] = 1
weights[Uw(ordW[0])[3]] = 1
weights[Uw(ordW[0])[4]] = 1
weights[Uw(ordW[0])[5]] = 2
weights[Uw(ordW[0])[6]] = 2
weights[Uw(ordW[0])[7]] = 2

# Let's make a 6 by 8 telling us to which u in U we jump
# Then let's make an 8 by 6 telling us to which w we jump from a given U. 

m1 = []
for w in ordW:
    newrow = []
    for u in U:
        if u in Uw(w):
            newrow.append((v - 1)^weights[u])
        else:
            newrow.append(0)
    m1.append(normalize(newrow))

m2 = []
for i in range(len(U)):
    m2.append(normalize(signature(U[i])))
    
m = matrix(m1)*matrix(m2)

#print(matrix(m1).subs(v=2))
#print()
#print(matrix(m2).subs(v=2))
#print(matrix(m).subs(v=13))
print(m)
print(matrix(m).subs(v=2))

"""
Optional: eigenvalue information.
subbed = m.subs(v=7)
print(subbed)
subbed.eigenvalues()

def eigs(n):
    subbed = m.subs(v=n)
    return subbed.eigenvalues()

print(subbed.eigenvectors_right())
"""

[                                          v^3/(v^3 + 2*v^2 + 2*v + 1)     (1/2*v^3 + v^2 + 1/2*v - 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)     (1/2*v^3 + v^2 + 1/2*v - 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)         (1/2*v^3 + 1/2*v^2 + 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)         (1/2*v^3 + 1/2*v^2 + 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)                                             1/(v^3 + 2*v^2 + 2*v + 1)]
[    (1/2*v^3 + v^2 + 1/2*v - 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2) (1/2*v^4 + v^3 + 1/2*v^2 - 1/2*v)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)         (1/2*v^3 + 1/2*v^2 + 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)       (1/2*v^4 + 1/2*v^3 + 1/2*v)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)                                             1/(v^3 + 2*v^2 + 2*v + 1)                                             v/(v^3 + 2*v^2 + 2*v + 1)]
[    (1/2*v^3 + v^2 + 1/2*v - 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2)         (1/2*v^3 + 1/2*v^2 + 1/2)/(v^4 + 5/2*v^3 + 3*v^2 + 2*v + 1/2) (1/2*v^4 + v^3 

'\nOptional: eigenvalue information.\nsubbed = m.subs(v=7)\nprint(subbed)\nsubbed.eigenvalues()\n\ndef eigs(n):\n    subbed = m.subs(v=n)\n    return subbed.eigenvalues()\n\nprint(subbed.eigenvectors_right())\n'

In [30]:
subbed = m.subs(v=2)
print(subbed)
subbed.eigenvalues()

def eigs(n):
    subbed = m.subs(v=n)
    return subbed.eigenvalues()

eigs = subbed.eigenvectors_right()
for eig in eigs:
    print(eig[0])
    for a in eig[1]:
        print(a)
    print()

[  8/21 17/105 17/105 13/105 13/105   1/21]
[17/105 34/105 13/105 26/105   1/21   2/21]
[17/105 13/105 34/105   1/21 26/105   2/21]
[13/105 26/105   1/21 31/105   2/21   4/21]
[13/105   1/21 26/105   2/21 31/105   4/21]
[  1/21   2/21   2/21   4/21   4/21   8/21]
1
(1, 1, 1, 1, 1, 1)

2/5
(0, 1, -1, 1, -1, 0)

0
(0, 1, -1, -1, 1, 0)

0.06116960254157935?
(1, 3.101845619315047?, 3.101845619315047?, -6.438842912293455?, -6.438842912293455?, 5.673994585956817?)

0.1678942879863615?
(1, -0.6933230821837668?, -0.6933230821837668?, -0.0457091118138524?, -0.0457091118138524?, 0.4780643879952383?)

0.3709361094720592?
(1, 0.3414774628687205?, 0.3414774628687205?, -0.265447975892693?, -0.265447975892693?, -1.152058973952056?)



In [35]:
J = matrix([[1,0,0,0,0,0],[0,0,1,0,0,0],[0,1,0,0,0,0],[0,0,0,0,1,0],[0,0,0,1,0,0],[0,0,0,0,0,1]])
print(J)
print(J*subbed - subbed)

[1 0 0 0 0 0]
[0 0 1 0 0 0]
[0 1 0 0 0 0]
[0 0 0 0 1 0]
[0 0 0 1 0 0]
[0 0 0 0 0 1]
[   0    0    0    0    0    0]
[   0 -1/5  1/5 -1/5  1/5    0]
[   0  1/5 -1/5  1/5 -1/5    0]
[   0 -1/5  1/5 -1/5  1/5    0]
[   0  1/5 -1/5  1/5 -1/5    0]
[   0    0    0    0    0    0]


In [36]:
def fixed_points_in_bruhat_cell(u, w):
    """
    Given u in U (unipotent upper triangular) and w in W (permutation matrix),
    returns the size of the intersection of fixed points of u on G/B with BwB.
    
    Parameters:
    - u: element of U (unipotent upper triangular matrix)
    - w: element of W (permutation matrix)
    
    Returns:
    - Integer: size of intersection
    """
    # Get fixed flags of u
    fixed_flags_u = []
    for g in orbit_dict:
        if g.inverse() * u * g in B:
            fixed_flags_u.append(g)
    
    # Intersect with Bruhat cell of w
    intersection = []
    for x in biorbit_dict[w]:
        if x in fixed_flags_u:
            intersection.append(x)
    
    return len(intersection)

In [41]:
def fixed_points_in_bruhat_cell(u, w):
    """
    Given u in U (unipotent upper triangular) and w in W (permutation matrix),
    returns the size of the intersection of fixed points of u on G/B with BwB.
    
    Parameters:
    - u: element of U (unipotent upper triangular matrix) - can be matrix or group element
    - w: element of W (permutation matrix) - can be matrix or group element
    
    Returns:
    - Integer: size of intersection
    """
    # Convert to group elements if needed
    if not (u in G):
        u = G(u)
    if not (w in G):
        w = G(w)
    
    # Get fixed flags of u
    fixed_flags_u = []
    for g in orbit_dict:
        if g.inverse() * u * g in B:
            fixed_flags_u.append(g)
    
    # Intersect with Bruhat cell of w
    intersection = []
    for x in biorbit_dict[w]:
        if x in fixed_flags_u:
            intersection.append(x)
    
    return len(intersection)

In [46]:
perm1 = matrix([[0,1,0],[1,0,0],[0,0,1]])
perm2 = matrix([[1,0,0],[0,0,1],[0,1,0]])
a = 
u = matrix([[1,0,1],[0,1,1],[0,0,1]])
print(u)
print(fixed_points_in_bruhat_cell(u, perm1))
print(fixed_points_in_bruhat_cell(u, perm2))
print(fixed_points_in_bruhat_cell(u, perm1*perm2))
print(fixed_points_in_bruhat_cell(u, perm2*perm1))

[1 0 1]
[0 1 1]
[0 0 1]
2
0
2
0


In [80]:
import random

# Generate a random 3x3 upper triangular matrix with entries in {0, 1}
def get_random_u():
    # Diagonal is all 1s for a standard upper triangular unipotent matrix
    u = matrix([
        [1, random.randint(0, 1), random.randint(0, 1)],
        [0, 1,                   random.randint(0, 1)],
        [0, 0,                   1]
    ])
    return u

# Define the permutations
perm1 = matrix([[0,1,0],[1,0,0],[0,0,1]])
perm2 = matrix([[1,0,0],[0,0,1],[0,1,0]])

# Set u to the random matrix
u = get_random_u()

print("Matrix u:")
print(u)
print("-" * 10)

# Computing fixed points (assuming the function is defined in your environment)
print(fixed_points_in_bruhat_cell(u, perm1) - fixed_points_in_bruhat_cell(u, perm2))
print(fixed_points_in_bruhat_cell(u, perm1*perm2) - fixed_points_in_bruhat_cell(u, perm2*perm1))

Matrix u:
[1 0 1]
[0 1 0]
[0 0 1]
----------
0
0


In [84]:
import numpy as np
from itertools import product

# --- Configuration ---
P = 5  # Finite field size (F_5)
np.set_printoptions(linewidth=200)

# --- 1. Group & Matrix Utilities ---

def mult(A, B):
    return np.matmul(A, B) % P

def invert(A):
    # Standard modular inverse using determinant and adjugate for 3x3
    det = int(round(np.linalg.det(A))) % P
    if det == 0: raise ValueError("Singular matrix")
    det_inv = pow(det, -1, P)
    
    # Adjugate matrix
    adj = np.zeros((3, 3), dtype=int)
    adj[0,0] = A[1,1]*A[2,2] - A[1,2]*A[2,1]
    adj[0,1] = A[0,2]*A[2,1] - A[0,1]*A[2,2]
    adj[0,2] = A[0,1]*A[1,2] - A[0,2]*A[1,1]
    adj[1,0] = A[1,2]*A[2,0] - A[1,0]*A[2,2]
    adj[1,1] = A[0,0]*A[2,2] - A[0,2]*A[2,0]
    adj[1,2] = A[0,2]*A[1,0] - A[0,0]*A[1,2]
    adj[2,0] = A[1,0]*A[2,1] - A[1,1]*A[2,0]
    adj[2,1] = A[0,1]*A[2,0] - A[0,0]*A[2,1]
    adj[2,2] = A[0,0]*A[1,1] - A[0,1]*A[1,0]
    
    return (adj * det_inv) % P

def is_upper_triangular(A):
    # Check strict lower triangle is 0
    return A[1,0] == 0 and A[2,0] == 0 and A[2,1] == 0

# --- 2. Weyl Group S3 Setup ---

# Permutation matrices
s1 = np.array([[0,1,0],[1,0,0],[0,0,1]])
s2 = np.array([[1,0,0],[0,0,1],[0,1,0]])
id_mat = np.eye(3, dtype=int)

perms = {
    "id":   id_mat,
    "s1":   s1,
    "s2":   s2,
    "s1s2": mult(s1, s2),
    "s2s1": mult(s2, s1),
    "w0":   mult(s1, mult(s2, s1))
}

# The Dynkin automorphism: s1 <-> s2
sigma_map = {
    "id": "id", 
    "s1": "s2", 
    "s2": "s1",
    "s1s2": "s2s1", 
    "s2s1": "s1s2", 
    "w0": "w0"
}

# --- 3. Cell Generation ---

def get_all_unipotents():
    # Returns all matrices in U(F_p)
    # Form: [[1, a, b], [0, 1, c], [0, 0, 1]]
    us = []
    for a, b, c in product(range(P), repeat=3):
        us.append(np.array([[1, a, b], [0, 1, c], [0, 0, 1]]))
    return us

def get_cell_flags(z_name):
    """
    Generates all flags in the Bruhat cell BzB.
    Representative approach: g = v * z, where v in U \cap z U^- z^{-1}.
    """
    z = perms[z_name]
    z_inv = z.T # Orthogonal for permutations
    
    flags = []
    
    # Brute force check: which u in U satisfy z^-1 u z is Lower Triangular?
    # This identifies the subgroup U \cap z U^- z^{-1}
    all_u = get_all_unipotents()
    
    for v in all_u:
        # Check if v is in the correct intersection
        conj = mult(mult(z_inv, v), z)
        # Check if conj is Lower Triangular (strict upper is 0)
        if conj[0,1] == 0 and conj[0,2] == 0 and conj[1,2] == 0:
            # v is a valid representative
            flags.append(mult(v, z))
            
    return flags

# --- 4. Main Experiment ---

def run_test():
    # 1. Pick a generic u in cell B s1 B
    # A generic element looks like I + c*E12 where c != 0
    u = np.eye(3, dtype=int)
    u[0,1] = 1 # Simple representative for s1 type
    
    print(f"Testing Conjecture for u in B(s1)B over F_{P}...")
    print(f"u =\n{u}\n")
    
    results = {}
    
    # 2. Iterate over all Weyl group elements z
    for z_name in perms.keys():
        flags = get_cell_flags(z_name)
        
        fixed_count = 0
        for flag in flags:
            # Check fixed point condition: flag^-1 * u * flag \in B
            # (i.e., is Upper Triangular)
            flag_inv = invert(flag)
            conj = mult(mult(flag_inv, u), flag)
            
            if is_upper_triangular(conj):
                fixed_count += 1
        
        results[z_name] = fixed_count

    # 3. Print Comparison Table
    print(f"{'z':<6} | {'|Xu n BzB|':<12} || {'sigma(z)':<8} | {'|Xu n Bsig(z)B|':<15} | {'Diff':<5}")
    print("-" * 65)
    
    checked_pairs = set()
    
    for z in perms.keys():
        sig_z = sigma_map[z]
        
        # Avoid printing duplicate symmetric pairs to keep table clean
        pair_key = tuple(sorted((z, sig_z)))
        if pair_key in checked_pairs and z != sig_z:
            continue
        checked_pairs.add(pair_key)
        
        count_z = results[z]
        count_sig = results[sig_z]
        diff = count_z - count_sig
        
        print(f"{z:<6} | {count_z:<12} || {sig_z:<8} | {count_sig:<15} | {diff:<5}")

if __name__ == "__main__":
    run_test()

Testing Conjecture for u in B(s1)B over F_5...
u =
[[1 1 0]
 [0 1 0]
 [0 0 1]]

z      | |Xu n BzB|   || sigma(z) | |Xu n Bsig(z)B| | Diff 
-----------------------------------------------------------------
id     | 1            || id       | 1               | 0    
s1     | 0            || s2       | 5               | -5   
s1s2   | 0            || s2s1     | 5               | -5   
w0     | 0            || w0       | 0               | 0    
